# GA Forget5 — single T4, no DeepSpeed

예전에 PO가 잘 재현된 실행 경로에 맞춘 GA 통제 실험입니다. 공개 Stage-I에서 새로 시작하며, T4 한 장과 Hugging Face Accelerate만 사용합니다. DeepSpeed와 분산 학습은 사용하지 않습니다.

고정 조건: Transformers 4.46.3, Tokenizers 0.20.3, PEFT 0.13.2, Accelerate 0.34.2, fast tokenizer, FP16 mixed precision, SDPA, batch 2, accumulation 64, effective forget batch 128, 8 epochs, GA LR 2e-5.


In [ ]:
%%bash
set -euo pipefail
cd /kaggle/working
if [ ! -d FIUBench_kaggle ]; then
  git clone https://github.com/npngjjm/FIUBench_kaggle.git
fi
python -m pip uninstall -y torchao deepspeed || true
python -m pip install --no-cache-dir --upgrade \
  'transformers==4.46.3' \
  'tokenizers==0.20.3' \
  'peft==0.13.2' \
  'accelerate==0.34.2' \
  'huggingface_hub==0.36.2' \
  'numpy==2.1.3' \
  hydra-core omegaconf datasets sentencepiece protobuf scipy pillow \
  pyyaml requests tqdm rouge_score scikit-learn
echo 'INSTALL_COMPLETE — 다음 셀에서 커널을 한 번 재시작하세요.'


설치 셀 완료 후 아래 셀을 한 번 실행합니다. 연결이 돌아오면 이 셀은 다시 실행하지 말고 환경 확인 셀부터 진행하세요.


In [ ]:
import os
os._exit(0)


In [ ]:
import gc, hashlib, json, math, os, shutil, sys, zipfile
from collections import defaultdict
from importlib.metadata import PackageNotFoundError, version
from pathlib import Path

import torch
from huggingface_hub import HfApi, hf_hub_download
from PIL import Image

repo = Path('/kaggle/working/FIUBench_kaggle')
assert repo.is_dir(), '저장소가 없습니다. 설치 셀을 다시 실행하세요.'
os.chdir(repo)
if str(repo) not in sys.path:
    sys.path.insert(0, str(repo))
os.environ['HF_HOME'] = '/kaggle/working/hf_cache'
os.environ['TOKENIZERS_PARALLELISM'] = 'false'
os.environ['PYTORCH_CUDA_ALLOC_CONF'] = 'expandable_segments:True'

expected = {
    'transformers': '4.46.3', 'tokenizers': '0.20.3', 'peft': '0.13.2',
    'accelerate': '0.34.2', 'huggingface-hub': '0.36.2', 'numpy': '2.1.3',
}
for package, wanted in expected.items():
    installed = version(package)
    print(package, installed)
    assert installed == wanted, (package, installed, wanted)
try:
    version('deepspeed')
    raise AssertionError('DeepSpeed가 설치되어 있습니다. 설치 셀부터 다시 실행하세요.')
except PackageNotFoundError:
    print('deepspeed: NOT INSTALLED')
assert torch.cuda.is_available(), 'Kaggle에서 GPU를 켜세요.'
assert torch.cuda.device_count() >= 1
print('torch:', torch.__version__)
print('사용 GPU:', torch.cuda.get_device_name(0))
print('보이는 GPU 수:', torch.cuda.device_count(), '(학습은 cuda:0 한 장만 사용)')
print('SINGLE_GPU_ENVIRONMENT_OK')


In [ ]:
# 공개 FIUBench revision에서 학습에 필요한 400명 이미지를 복원합니다.
records = [json.loads(s) for s in (repo / 'dataset/full.json').read_text().splitlines() if s.strip()][:400]
needed = {row['image_path'] for row in records}
assert len(needed) == 400
dataset_revision = HfApi().dataset_info('gray311/FIUBench').sha
archive = hf_hub_download(
    'gray311/FIUBench', 'SFHQ.zip', repo_type='dataset', revision=dataset_revision,
    local_dir='/kaggle/working/fiu_single_gpu_download',
)
with zipfile.ZipFile(archive) as zipped:
    entries = defaultdict(list)
    for member in zipped.infolist():
        path = Path(member.filename)
        if not member.is_dir() and '__MACOSX' not in path.parts and not path.name.startswith('._'):
            entries[path.name].append(member)
    for image_path in needed:
        destination = (repo / image_path).resolve()
        assert destination.is_relative_to((repo / 'dataset').resolve())
        matches = entries[destination.name]
        assert len(matches) == 1, (destination.name, len(matches))
        destination.parent.mkdir(parents=True, exist_ok=True)
        with zipped.open(matches[0]) as source, destination.open('wb') as target:
            shutil.copyfileobj(source, target)
        with Image.open(destination) as image:
            image.verify()
assert all((repo / path).is_file() for path in needed)
Path('/kaggle/working/fiu_ga_single_gpu_data_revision.txt').write_text(dataset_revision + '\n')
print('이미지 검증:', len(needed), 'dataset revision:', dataset_revision)
print('SINGLE_GPU_DATA_OK')


In [ ]:
# 학습 config와 400개 forget QA의 tokenizer/label 경계를 고정합니다.
from omegaconf import OmegaConf
from transformers import AutoTokenizer
from data_module import preprocess_v1
from utils import get_model_identifiers_from_yaml

output_dir = Path('/kaggle/working/models/llava_phi_GA_forget5_single_t4_no_deepspeed')
assert not output_dir.exists() or not any(output_dir.iterdir()), (
    f'기존 결과 보호: {output_dir}. 이 노트북을 중복 실행하지 마세요.'
)
output_dir.mkdir(parents=True, exist_ok=True)

cfg = OmegaConf.create({
    'model_family': 'llava-phi',
    'model_path': 'gray311/vlm_unlearning_ft_llava_phi_3_mini',
    'attn_implementation': 'sdpa',
    'data_path': './dataset/full.json', 'split': 'forget5', 'forget_loss': 'ga',
    'lr': 2e-5, 'batch_size': 2, 'gradient_accumulation_steps': 64,
    'num_epochs': 8, 'seed': 233, 'max_grad_norm': 1.0,
    'lr_scheduler_type': 'cosine', 'warmup_ratio': 0.06,
    'LoRA': {'r': 128, 'alpha': 256, 'dropout': 0.05},
    'tune_vision_tower': False, 'tune_mm_projector': True, 'tune_language_model': True,
    'workers': 0, 'save_dir': str(output_dir), 'save_steps': 4,
})
OmegaConf.save(cfg, output_dir / 'cfg.yaml')

tokenizer = AutoTokenizer.from_pretrained(cfg.model_path, use_fast=True)
assert tokenizer.is_fast and tokenizer.__class__.__name__ == 'LlamaTokenizerFast'
model_cfg = get_model_identifiers_from_yaml(cfg.model_family)
question_tag, answer_tag = model_cfg['question_start_tag'], model_cfg['answer_tag']
roles = [question_tag, answer_tag]
split = json.loads((repo / 'dataset/split.json').read_text())
forget_ids = set(split['forget5'])
samples = [qa for row in records if row['unique_id'] in forget_ids for qa in row['qa_list']]
assert len(forget_ids) == 20 and len(samples) == 400
digest = hashlib.sha256()
assistant_id = tokenizer.convert_tokens_to_ids('<|assistant|>')
for qa in samples:
    conversation = (question_tag + '<image>\n' + qa['question'].capitalize()
                    + answer_tag + qa['answer'].capitalize())
    encoded = tokenizer(conversation, max_length=512, truncation=True, return_tensors='pt')
    labels = preprocess_v1(tokenizer, encoded['input_ids'], conversation, roles)
    supervised = labels[0][labels[0] != -100]
    assert supervised.numel() > 1 and int(supervised[0]) == assistant_id
    digest.update(encoded['input_ids'].numpy().tobytes())
    digest.update(labels.numpy().tobytes())
token_signature = digest.hexdigest()
(output_dir / 'training_token_signature_512.txt').write_text(token_signature + '\n')
print(OmegaConf.to_yaml(cfg))
print('token signature:', token_signature)
print('effective forget batch:', cfg.batch_size * cfg.gradient_accumulation_steps)
assert cfg.batch_size * cfg.gradient_accumulation_steps == 128
print('SINGLE_GPU_GA_PREFLIGHT_OK')


In [ ]:
# 모델·LoRA·optimizer를 준비합니다. 이 셀은 한 번만 실행하세요.
from accelerate import Accelerator
from accelerate.utils import set_seed
from peft import LoraConfig, get_peft_model
from torch.utils.data import DataLoader
from tqdm.auto import tqdm
from transformers import CLIPImageProcessor, LlavaForConditionalGeneration, get_scheduler
from data_module import MMForgetDatasetQA, custom_data_collator_forget
from utils import save_lora_weights

set_seed(cfg.seed)
accelerator = Accelerator(
    gradient_accumulation_steps=cfg.gradient_accumulation_steps,
    mixed_precision='fp16',
)
assert accelerator.num_processes == 1
assert accelerator.distributed_type.name == 'NO'
print(accelerator.state)

image_processor = CLIPImageProcessor.from_pretrained('openai/clip-vit-large-patch14-336')
model = LlavaForConditionalGeneration.from_pretrained(
    cfg.model_path, attn_implementation='sdpa', torch_dtype=torch.float16,
)
model.config.use_cache = False
model.gradient_checkpointing_enable()
model.enable_input_require_grads()
target_modules = r'.*language_model.*\.(up_proj|k_proj|linear_2|down_proj|v_proj|q_proj|o_proj|gate_proj|linear_1)'
model = get_peft_model(model, LoraConfig(
    r=cfg.LoRA.r, lora_alpha=cfg.LoRA.alpha, target_modules=target_modules,
    lora_dropout=cfg.LoRA.dropout, bias='none', task_type='CAUSAL_LM',
))
for name, parameter in model.named_parameters():
    if cfg.tune_mm_projector and ('projector' in name or 'multi_modal_projector' in name):
        parameter.requires_grad = True
# 예전 단일-T4 PO 경로와 같이 학습 가능 FP16 파라미터를 FP32로 유지합니다.
for parameter in model.parameters():
    if parameter.requires_grad and parameter.dtype == torch.float16:
        parameter.data = parameter.data.float()
trainable = {name: parameter for name, parameter in model.named_parameters() if parameter.requires_grad}
assert len(trainable) == 452, len(trainable)
assert all(parameter.dtype == torch.float32 for parameter in trainable.values())
print('trainable tensors:', len(trainable))
print('trainable parameters:', sum(p.numel() for p in trainable.values()))

dataset = MMForgetDatasetQA(
    config=cfg, tokenizer=tokenizer, image_processor=image_processor,
    max_length=512, processor=None,
)
assert len(dataset) == 400
dataloader = DataLoader(
    dataset, batch_size=cfg.batch_size, num_workers=0, shuffle=False,
    collate_fn=custom_data_collator_forget(tokenizer=tokenizer),
)
assert len(dataloader) == 200
optimizer = torch.optim.AdamW(
    [
        {'params': [p for n, p in trainable.items() if 'bias' not in n], 'weight_decay': 0.01},
        {'params': [p for n, p in trainable.items() if 'bias' in n], 'weight_decay': 0.0},
    ],
    lr=cfg.lr,
)
attempts_per_epoch = math.ceil(len(dataloader) / cfg.gradient_accumulation_steps)
max_attempts = cfg.num_epochs * attempts_per_epoch
assert attempts_per_epoch == 4 and max_attempts == 32
warmup_steps = round(cfg.warmup_ratio * max_attempts)
scheduler = get_scheduler(
    cfg.lr_scheduler_type, optimizer=optimizer, num_warmup_steps=warmup_steps,
    num_training_steps=max_attempts,
)
model, optimizer, dataloader = accelerator.prepare(model, optimizer, dataloader)
assert len(dataloader) == 200
print('microbatches/epoch:', len(dataloader))
print('update attempts/epoch:', attempts_per_epoch, 'total:', max_attempts)
print('warmup steps:', warmup_steps)
print('SINGLE_GPU_MODEL_AND_OPTIMIZER_OK')


In [ ]:
# 이 셀 한 번으로 8 epochs를 실행합니다. 실행 중 다시 누르지 마세요.
audit_path = output_dir / 'optimizer_scheduler_audit.jsonl'
assert not audit_path.exists(), f'기존 감사 로그 보호: {audit_path}'
attempts = successes = skips = 0
records_out = []
progress = tqdm(total=max_attempts, desc='GA update attempts')

for epoch in range(cfg.num_epochs):
    model.train()
    for microbatch, batch in enumerate(dataloader):
        forget_inputs, _retain_inputs = batch
        with accelerator.accumulate(model):
            lr_used = [float(group['lr']) for group in optimizer.param_groups]
            outputs = model(**forget_inputs)
            loss = -outputs.loss
            assert torch.isfinite(loss).item(), (epoch, microbatch, float(loss.detach()))
            accelerator.backward(loss)
            if accelerator.sync_gradients:
                accelerator.clip_grad_norm_(model.parameters(), cfg.max_grad_norm)
            optimizer.step()
            skipped = bool(accelerator.optimizer_step_was_skipped) if accelerator.sync_gradients else False
            if accelerator.sync_gradients and not skipped:
                scheduler.step()
            optimizer.zero_grad()

        if accelerator.sync_gradients:
            attempts += 1
            if skipped:
                skips += 1
            else:
                successes += 1
            assert attempts == successes + skips
            assert attempts <= max_attempts
            lr_next = [float(group['lr']) for group in optimizer.param_groups]
            record = {
                'epoch': epoch, 'microbatch': microbatch, 'attempt': attempts,
                'successful_updates': successes, 'skipped_updates': skips,
                'skipped': skipped, 'scheduler_index': int(scheduler.last_epoch),
                'lr_used': lr_used, 'lr_next': lr_next,
                'ga_loss': float(loss.detach().float().cpu()),
            }
            assert record['scheduler_index'] == successes
            records_out.append(record)
            with audit_path.open('a') as stream:
                stream.write(json.dumps(record, allow_nan=False) + '\n')
            progress.update(1)
            progress.set_postfix(success=successes, skip=skips, lr=f'{lr_next[0]:.2e}')
            if attempts % cfg.save_steps == 0:
                destination = output_dir / f'step_{attempts}'
                destination.mkdir(parents=True, exist_ok=False)
                save_lora_weights(accelerator.unwrap_model(model), destination)
                gc.collect()
                torch.cuda.empty_cache()

progress.close()
assert attempts == max_attempts == 32
save_lora_weights(accelerator.unwrap_model(model), output_dir)
accelerator.wait_for_everyone()
print('TRAINING_COMPLETE attempts=', attempts, 'successful=', successes, 'skipped=', skips)


In [ ]:
# 로그, LR 진행, 9개 checkpoint 및 최종 텐서를 검증합니다.
audit = [json.loads(line) for line in audit_path.read_text().splitlines()]
assert len(audit) == 32
for index, record in enumerate(audit, 1):
    assert record['attempt'] == index
    assert record['successful_updates'] + record['skipped_updates'] == index
    assert record['scheduler_index'] == record['successful_updates']
checkpoint_paths = [output_dir / f'step_{step}' / 'checkpoint.pt' for step in range(4, 33, 4)]
checkpoint_paths.append(output_dir / 'checkpoint.pt')
for path in checkpoint_paths:
    assert path.is_file(), path
    state = torch.load(path, map_location='cpu', weights_only=True)
    assert len(state) == 452, (path, len(state))
    assert sum('lora_' in key for key in state) == 448
    assert sum('projector' in key for key in state) == 4
    assert all(torch.isfinite(tensor).all().item() for tensor in state.values())
    del state
print('epoch | attempt | successful | skips | scheduler | LR used -> next')
for record in audit:
    if record['microbatch'] == 199:
        print(f"{record['epoch']:5d} | {record['attempt']:7d} | "
              f"{record['successful_updates']:10d} | {record['skipped_updates']:5d} | "
              f"{record['scheduler_index']:9d} | {record['lr_used'][0]:.6e} -> {record['lr_next'][0]:.6e}")
print('최종:', audit[-1])
print('final checkpoint:', output_dir / 'checkpoint.pt')
print('SINGLE_GPU_GA_TRAINING_AND_CHECKPOINT_AUDIT_OK')


학습과 감사가 끝나면 `Save Version`으로 Output을 보존하세요. 이후 `FIUBench_forget5_fast_eval_kaggle.ipynb`에 이 저장 Output 하나만 연결하고 `METHOD = "GA"`로 Core 평가를 실행합니다. 먼저 마지막 표와 `SINGLE_GPU_GA_TRAINING_AND_CHECKPOINT_AUDIT_OK`를 캡처해 두세요.
